# Session 03 — Generative Adversarial Networks (GANs)**Framework:** PyTorch

## Objectives1. Build a `simple_generator()` function.2. Implement a `Discriminator` class.3. Run an adversarial training loop simulation.4. Visualise generator and discriminator losses.5. Compare GAN variants through research.

## Theory### Generative Adversarial NetworksA GAN consists of two networks trained simultaneously:- **Generator G**: maps noise $z \sim \mathcal{N}(0, I)$ to fake data.- **Discriminator D**: classifies inputs as real or fake.The training objective is a minimax game:$$\min_G \max_D \; \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$```mermaidgraph LR    Z[Noise z] --> G[Generator]    G --> FD[Fake Data]    RD[Real Data] --> D[Discriminator]    FD --> D    D --> P[Real / Fake?]    style G fill:#2ecc71,stroke:#333    style D fill:#e74c3c,stroke:#333```

## Learning Outcomes- Implement generator and discriminator architectures.- Understand the adversarial training dynamic.- Interpret GAN loss curves.- Compare GAN variants (DCGAN, WGAN, StyleGAN).

---## Setup

In [ ]:
import sysfrom pathlib import PathPROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()if str(PROJECT_ROOT) not in sys.path:    sys.path.insert(0, str(PROJECT_ROOT))import torchimport matplotlib.pyplot as pltimport pandas as pdimport numpy as npfrom src.utils import seed_everything, get_device, setup_loggingfrom src.config import ensure_dirsfrom src.gan import (    simple_generator, Discriminator, Generator,    adversarial_loop_simulation, generate_gan_samples,)from src.visualization import plot_image_grid, save_generated_imagessetup_logging()seed_everything(42)ensure_dirs()device = get_device()print(f'Device: {device}')

---## 1. simple_generator()

In [ ]:
gen_simple = simple_generator(latent_dim=16, output_dim=784)print(gen_simple)# Test forward passz_test = torch.randn(4, 16)out = gen_simple(z_test)print(f'\nInput shape:  {z_test.shape}')print(f'Output shape: {out.shape}')print(f'Output range: [{out.min().item():.3f}, {out.max().item():.3f}]')

In [ ]:
# Visualise random generator output (untrained — expected to be noise)random_images = (out.detach() + 1) / 2  # Tanh → [0,1]plot_image_grid(random_images.view(-1, 1, 28, 28), nrow=4,                title='Untrained Generator Output (Random Noise)')

---## 2. Discriminator Class

In [ ]:
disc = Discriminator(input_dim=784)print(disc)# Test forward passfake_input = torch.randn(4, 784)d_out = disc(fake_input)print(f'\nInput shape:  {fake_input.shape}')print(f'Output shape: {d_out.shape}')print(f'Probabilities: {d_out.detach().squeeze().tolist()}')

---## 3. Adversarial Loop Simulation

In [ ]:
gen = Generator(latent_dim=16, output_dim=784)disc = Discriminator(input_dim=784)history = adversarial_loop_simulation(    generator=gen,    discriminator=disc,    device=device,    latent_dim=16,    steps=200,    batch_size=64,    lr=2e-4,)print(f'\nTraining complete — {len(history["d_losses"])} steps')

In [ ]:
# Plot GAN lossesfig, ax = plt.subplots(figsize=(10, 4))ax.plot(history['d_losses'], label='Discriminator Loss', alpha=0.7)ax.plot(history['g_losses'], label='Generator Loss', alpha=0.7)ax.set_xlabel('Step')ax.set_ylabel('Loss')ax.set_title('GAN Training — Adversarial Loss Curves')ax.legend()ax.grid(True, alpha=0.3)fig.tight_layout()plt.show()

### Interpretation- The discriminator loss should decrease as it gets better at classifying real vs fake.- The generator loss should decrease as its outputs fool the discriminator more often.- In an ideal equilibrium both losses stabilise around $\log 2 \approx 0.693$.- This simulation uses synthetic data so convergence patterns differ from real datasets.

In [ ]:
# Generate and save samples from the trained generatorsamples = generate_gan_samples(gen, device, n=16, latent_dim=16)plot_image_grid(    torch.tensor(samples).unsqueeze(1), nrow=8,    title='GAN Generated Samples (after training simulation)')saved = save_generated_images(torch.tensor(samples).unsqueeze(1), prefix='gan_generated')print(f'Saved {len(saved)} GAN samples')

---## 4. GAN Comparison Research

In [ ]:
gan_comparison = pd.DataFrame({    'GAN Variant': ['Vanilla GAN', 'DCGAN', 'WGAN', 'WGAN-GP', 'StyleGAN', 'CycleGAN', 'Pix2Pix'],    'Year': [2014, 2016, 2017, 2017, 2019, 2017, 2017],    'Key Innovation': [        'Original adversarial framework',        'Convolutional architecture with batch norm',        'Wasserstein distance as loss',        'Gradient penalty instead of clipping',        'Style-based generator with AdaIN',        'Unpaired image-to-image translation',        'Paired image-to-image translation',    ],    'Architecture': [        'MLP',        'CNN (strided convolutions)',        'CNN + weight clipping',        'CNN + gradient penalty',        'Progressive growing CNN',        'Two generator-discriminator pairs',        'U-Net generator + PatchGAN disc.',    ],    'Best For': [        'Proof of concept',        'Image generation',        'Stable training',        'High-quality image generation',        'Face synthesis',        'Domain transfer (horse↔zebra)',        'Supervised image translation',    ],})gan_comparison.index = range(1, len(gan_comparison) + 1)gan_comparison.index.name = '#'gan_comparison

In [ ]:
# Visualise GAN timelinefig, ax = plt.subplots(figsize=(12, 3))years = gan_comparison['Year']names = gan_comparison['GAN Variant']colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(names)))for i, (y, n) in enumerate(zip(years, names)):    ax.scatter(y, 0.5, c=[colors[i]], s=150, zorder=5)    offset = 15 if i % 2 == 0 else -20    ax.annotate(n, (y, 0.5), textcoords='offset points',               xytext=(0, offset), ha='center', fontsize=9)ax.set_xlim(2013, 2020)ax.set_yticks([])ax.set_title('Evolution of GAN Architectures', fontsize=13)ax.grid(axis='x', alpha=0.3)fig.tight_layout()plt.show()

---## Summary- Built a `simple_generator()` and a `Discriminator` class.- Ran an adversarial training simulation and visualised the loss dynamics.- Compared 7 major GAN variants across architecture, innovation, and use case.- Observed that GAN training is inherently unstable — various tricks (WGAN, spectral norm) improve it.

---## Interview Questions1. Explain the minimax objective of GANs.2. What is mode collapse and how can it be detected?3. Why was the Wasserstein distance introduced in WGAN?4. Compare DCGAN and StyleGAN architectures.5. How does a conditional GAN differ from a vanilla GAN?6. What is the gradient penalty in WGAN-GP?7. Can GANs be used for data augmentation? Explain.8. What metrics (FID, IS) are used to evaluate GAN quality?

---## References1. Goodfellow, I. et al. (2014). *Generative Adversarial Nets*. NeurIPS.2. Radford, A. et al. (2016). *Unsupervised Representation Learning with DCGANs*. ICLR.3. Arjovsky, M. et al. (2017). *Wasserstein GAN*. ICML.4. Karras, T. et al. (2019). *A Style-Based Generator Architecture for GANs*. CVPR.5. Zhu, J. et al. (2017). *Unpaired Image-to-Image Translation using Cycle-Consistent GANs*. ICCV.